## 0. Import libraries

In [1]:
import os
import json
import torch
import wandb
from dotenv import load_dotenv
from datasets import * 
from pathlib import Path
from prompts import SYSTEM_PROMPT
from collections import defaultdict
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, Trainer, DataCollatorForSeq2Seq
from transformers.integrations import WandbCallback

load_dotenv()

/venv/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
wandb.login()

wandb.init(
    project="qwen2.5-7b-ft",
    name='run-1'
)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: lamyeungkong0108 (lamyeungkong0108-the-hong-kong-university-of-science-and) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## 1. Dataset Loading

In [ ]:
def convert_to_json(file_path: str, json_path):
    """
    Converting raw dataset to json format.
    For each data:
    {
        "sentence": the_sentence,
        "entity_info": [
            {
                "entity_text": the_substring_1,
                "entity_label": the_corresponding_label_1         
            },
            {
                "entity_text": the_substring_2,
                "entity_label": the_corresponding_label_2         
            },
            ...
        ]
    }
    """
    current_sub = ""
    current_sentences = ""
    current_dict = []
    current_label = None
    entities_label = []

    json_data = []

    print("Extracting data...")
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line: # Empty line -> last sentence has finished
                if current_label is not None and current_sub != "":
                    current_dict.append(
                        {
                            "entity_text": current_sub,
                            "entity_label": current_label
                        }
                    )   
                if current_sentences:          
                    json_data.append(
                        {
                            "sentence": current_sentences,
                            "entity_info": current_dict
                        }
                    )        
                current_sub = ""
                current_sentences = ""
                current_dict = []
                current_label = None                

            else:
                chunk = line.split(" ") # e.g.灵 I-方剂
                if len(chunk) == 2:
                    if len(chunk[1]) > 1:
                        tag, label = chunk[1].split('-')
                    else:
                        tag = chunk[1]
                        label = None

                    current_sentences += chunk[0]
                    if label and label == current_label:
                        current_sub += chunk[0]
                    elif tag == 'B':
                        if current_label is not None and current_sub != "":
                            current_dict.append(
                                {
                                    "entity_text": current_sub,
                                    "entity_label": current_label
                                }
                            )

                        # Assign new label 
                        current_label = label
                        current_sub = chunk[0]

                        if current_label not in entities_label:
                            entities_label.append(current_label)
                    else: # tag == 'O'
                        if current_label is not None and current_sub != "":
                            current_dict.append(
                                {
                                    "entity_text": current_sub,
                                    "entity_label": current_label
                                }
                            )
                        current_label = None
                        current_sub = ""
                else:
                    raise Exception(f"Chunk number is incorrect: {chunk}")
    print(f"Converting {len(json_data)} sentences with their entity information to a jsonal file.")
    with open(json_path, 'a', encoding='utf-8') as f:
        for obj in json_data:
            f.write(json.dumps(obj, ensure_ascii=False) + "\n")
    print("Convert Successfully.")
    return entities_label if entities_label else []
                    

train_dataset_path = "./data/medical.train"
train_json_dataset_path = "./data/medical_train.jsonl"
train_json_dataset_path = Path(train_json_dataset_path)
train_json_dataset_path.touch()
train_entities_label = convert_to_json(train_dataset_path, train_json_dataset_path)

test_data = "./data/medical.test"
test_json_dataset_path = "./data/medical_test.jsonl"
test_json_dataset_path = Path(test_json_dataset_path)
test_json_dataset_path.touch()
test_entities_label = convert_to_json(test_data, test_json_dataset_path)

val_data = "./data/medical.dev"
val_json_dataset_path = "./data/medical_val.jsonl"
val_json_dataset_path = Path(val_json_dataset_path)
val_json_dataset_path.touch()
val_entities_label = convert_to_json(val_data, val_json_dataset_path)

print(f"All unique entities in training dataset: {train_entities_label}")
print(f"All unique entities in testing dataset: {test_entities_label}")
print(f"All unique entities in valid dataset: {val_entities_label}")

train_entities_label = train_entities_label or []
test_entities_label = test_entities_label or []
val_entities_label = val_entities_label or []
entities_label = list(set(train_entities_label + test_entities_label + val_entities_label))
print(f"All unique entities combined: {entities_label}")

train_dataset = Dataset.from_json(str(train_json_dataset_path))
test_dataset = Dataset.from_json(str(test_json_dataset_path))
val_dataset = Dataset.from_json(str(val_json_dataset_path))

Extracting data...
Converting 5258 sentences with their entity information to a jsonal file.
Convert Successfully.
Extracting data...
Converting 657 sentences with their entity information to a jsonal file.
Convert Successfully.
Extracting data...
Converting 656 sentences with their entity information to a jsonal file.
Convert Successfully.
All unique entities in training dataset: ['临床表现', '中医治疗', '西医诊断', '方剂', '中药', '中医诊断', '西医治疗', '中医证候', '中医治则', '其他治疗']
All unique entities in testing dataset: ['中医诊断', '西医治疗', '西医诊断', '中医治则', '中医治疗', '临床表现', '中医证候', '方剂', '中药', '其他治疗']
All unique entities in valid dataset: ['方剂', '中药', '西医诊断', '中医治疗', '中医证候', '其他治疗', '临床表现', '中医治则', '西医治疗', '中医诊断']
All unique entities combined: ['中医诊断', '中药', '中医证候', '中医治则', '其他治疗', '西医诊断', '临床表现', '中医治疗', '西医治疗', '方剂']


AttributeError: 'PosixPath' object has no attribute 'decode'

In [ ]:
train_dataset[0]

## 2. Model Download 

In [ ]:
model_id = "Qwen/Qwen2.5-7B"    

tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=False, trust_remote_code=False)
tokenizer.pad_token_id = tokenizer.eos_token_id
model = AutoModelForCausalLM.from_pretrained(
    model_id, device_map="auto", 
    torch_dtype=torch.bfloat16,
    quantization_config=BitsAndBytesConfig
        (
            load_in_4bit=True, 
            bnb_4bit_compute_dtype=torch.bfloat16, 
            bnb_4bit_quant_type='nf4',
            bnb_4bit_use_double_quant=False 
        )
    )
model.enable_input_require_grads()

## 3. Data Preprocess

In [ ]:
# <|im_start|>system
# You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
# <|im_start|>user
# What is 2 + 2?<|im_end|>
# <|im_start|>assistant

def process_func(example):
    """
    Convert dataset from a json file to a format that is acceptable to model.
    """

    MAX_LENGTH = 512 
    input_ids, attention_mask, labels = [], [], []
    system_prompt = SYSTEM_PROMPT.format(entities=entities_label)

    instruction = tokenizer(
        f"<|im_start|>system\n{system_prompt}<|im_end|>\n<|im_start|>user\n{example['sentence']}<|im_end|>\n<|im_start|>assistant\n",
        add_special_tokens=False
    )
    response = tokenizer(f"{json.dumps(example['entity_info'], ensure_ascii=False)}", add_special_tokens=False)

    input_ids = instruction['input_ids'] + response['input_ids'] + [tokenizer.eos_token_id]
    attention_mask = instruction['attention_mask'] + response['attention_mask'] + [1]
    labels = [-100] * len(instruction['input_ids']) + response['input_ids'] + [tokenizer.eos_token_id]
    if len(input_ids) > MAX_LENGTH:
        input_ids = input_ids[:MAX_LENGTH]
        attention_mask = attention_mask[:MAX_LENGTH]
        labels = labels[:MAX_LENGTH]
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels                                                   
            }

train_dataset = train_dataset.map(process_func, remove_columns=train_dataset.column_names)
test_dataset = test_dataset.map(process_func, remove_columns=test_dataset.column_names)
val_dataset = val_dataset.map(process_func, remove_columns=val_dataset.column_names)

## 4. Setup Lora 

In [ ]:
for name, parameter in model.named_parameters():
    print(name)

In [ ]:
config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none"
)

## 5. Training 

In [ ]:
model = get_peft_model(model, config)

args = TrainingArguments(
    output_dir="./output/Qwen2.5-7B",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    logging_steps=10,
    num_train_epochs=2,
    save_steps=100,
    learning_rate=1e-4,
    save_on_each_node=True,
    gradient_checkpointing=True,
    report_to="none",
    eval_strategy="steps",
    eval_steps=50,    
)

In [ ]:
trainer = Trainer(
    model=model, 
    args=args,
    train_dataset=train_dataset, 
    eval_dataset=val_dataset,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
    callbacks=[WandbCallback()]
)

trainer.train()

In [ ]:
wandb.finish()

## 6. Evaluation 

In [ ]:
def calculate_f1(preds, truths):
    """
    preds: List of List of dicts. e.g. [[{"entity_text": the_substring_1, "entity_label": the_corresponding_label_1 }, ...], ...]
    truths: List of List of dicts. e.g. [[{"entity_text": the_substring_1, "entity_label": the_corresponding_label_1 }, ...], ...]
    """
    metrics = defaultdict(lambda: {'tp':0, 'fp':0, 'fn':0})

    for pred_list, gt_list in zip(preds, truths):

        all_labels = set([x["entity_label"] for x in pred_list] + [x["entity_label"] for x in gt_list])

        for label in all_labels:
            pred_label = {x["entity_text"] for x in pred_list if x["entity_label"] == label}
            truth_label = {x["entity_text"] for x in gt_list if x["entity_label"] == label}

            tp = len(pred_label & truth_label)
            fp = len(pred_label - truth_label)
            fn = len(truth_label - pred_label)

            metrics[label]['tp'] += tp
            metrics[label]['fp'] += fp
            metrics[label]['fn'] += fn

    total_tp = 0
    total_fp = 0 
    total_fn = 0

    for label, counts in metrics.items():
        tp, fp, fn = counts['tp'], counts['fp'], counts['fn']

        precision = tp / (tp + fp) if (tp + fp) != 0 else 0.0 
        recall = tp / (tp + fn) if (tp + fn) != 0 else 0.0 
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

        print(f"{label:<15} {precision:.4f}     {recall:.4f}     {f1:.4f}")

        total_tp += tp
        total_fp += fp 
        total_fn += fn

    micro_precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) != 0 else 0.0
    micro_recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) != 0 else 0.0 
    micro_f1 = 2 * micro_precision * micro_recall / (micro_precision + micro_recall) if (micro_precision + micro_recall) > 0 else 0.0
    print("-" * 45)
    print(f"{'Overall (Micro)':<15} {micro_precision:.4f}     {micro_recall:.4f}     {micro_f1:.4f}")

def predict(messages, model, tokenizer):
    device = "cuda"
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt").to(device)
    generated_ids = model.generate(model_inputs.input_ids, max_new_tokens=512)
    generated_ids = [
        output_ids[len(input_ids) :]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return response
    
preds = []
truths = []
for index, row in enumerate(test_dataset):
    sentence = row["sentence"]
    entity_info = row["entity_info"]
    try:
        truths.append(entity_info)
    except json.JSONDecodeError as e:
        print(f"{e}: {entity_info}")

    messages = [
        {"role": "system", "content": f"{SYSTEM_PROMPT.format(entities=entities_label)}"},
        {"role": "user", "content": f"{sentence}"},
    ]

    response = predict(messages, model, tokenizer)
    print(f"{messages[0]}\n\n{messages[1]}\n\n{response}")
    try:
        preds.append(json.loads(response))
    except json.JSONDecodeError as e:
        print(f"{e}: {response}")

calculate_f1(preds, truths)